# 05 - Neuron And MLP

这一节把前面的 `Value` 连接到神经网络。

这一节只抓一句话：

> 神经元就是很多个 `w * x` 加起来，再加一个 `b`，最后可选过 ReLU。

你会看到：

```text
Value -> Neuron -> Layer -> MLP
```

先不要急着训练。先看前向怎么把很多 Value 拼成一个预测值，再看 backward 怎么把梯度传回每个参数。

<!-- xiao-preview:start -->
## 课前预习作业：先数参数

神经网络这一节最容易迷糊的是“参数从哪里来”。先数 3 个小例子，再看 `Neuron`、`Layer`、`MLP` 的源码。


In [1]:
# TODO: 把 None 改成你数出来的参数个数。
preview_neuron_params = None   # 一个 Neuron，有 2 个输入：2 个 w + 1 个 b
preview_layer_params = None    # Layer(2, 3)：3 个神经元，每个 3 个参数
preview_mlp_params = None      # MLP(2, [3, 1])：先 Layer(2,3)，再 Layer(3,1)


def qa_check_05_preview(neuron_params, layer_params, mlp_params):
    values = [neuron_params, layer_params, mlp_params]
    if any(v is None for v in values):
        print('请先填写 TODO：先数清楚参数个数。')
        return
    assert neuron_params == 3, f'单个 Neuron 应该有 3 个参数，但你填的是 {neuron_params}'
    assert layer_params == 9, f'Layer(2, 3) 应该有 9 个参数，但你填的是 {layer_params}'
    assert mlp_params == 13, f'MLP(2, [3, 1]) 应该有 13 个参数，但你填的是 {mlp_params}'
    print('OK: 参数计数通过，可以进入神经网络结构。')


qa_check_05_preview(preview_neuron_params, preview_layer_params, preview_mlp_params)


请先填写 TODO：先数清楚参数个数。


<!-- xiao-preview:hint -->
<details>
<summary><strong>Show / Hide 提示</strong></summary>

一个神经元参数数 = 输入个数 + 1 个 bias。  
一层的参数数 = 神经元个数 * 每个神经元参数数。

</details>


<!-- xiao-preview:answer -->
<details>
<summary><strong>Show / Hide 答案</strong></summary>

```python
preview_neuron_params = 3
preview_layer_params = 9
preview_mlp_params = 13
```

</details>


## 0. Setup

In [2]:
from pathlib import Path
import sys
import random
import inspect

cwd = Path.cwd().resolve()
candidates = [cwd, cwd.parent, cwd / 'micrograd', cwd.parent / 'micrograd']
PROJECT_ROOT = None
for candidate in candidates:
    if (candidate / 'micrograd' / 'engine.py').exists():
        PROJECT_ROOT = candidate
        break

if PROJECT_ROOT is None:
    raise RuntimeError(f'Could not find local micrograd package from {cwd}')

sys.path.insert(0, str(PROJECT_ROOT))

from micrograd.engine import Value
from micrograd.nn import Neuron, Layer, MLP

random.seed(42)
print('project root:', PROJECT_ROOT)


def qa_check_count_mlp_parameters(func):
    sample = func(2, [3, 1])
    if sample is None:
        print('TODO: 请先实现 count_mlp_parameters，再运行本 cell。')
        return

    cases = [(2, [3, 1]), (2, [4, 1]), (3, [4, 4, 1])]
    for nin, nouts in cases:
        actual = func(nin, nouts)
        expected = len(MLP(nin, nouts).parameters())
        assert actual == expected, f'MLP({nin}, {nouts}): expected {expected}, got {actual}'
    print('OK: count_mlp_parameters passed.')


project root: /Users/barry/IdeaProjects/llm/micrograd


## 1. 先手写一个神经元

公式：

$$
n = ReLU(w_1x_1 + w_2x_2 + b)
$$

先用固定小数字，不用随机数。

In [3]:
x1 = Value(2.0)
x2 = Value(-3.0)
w1 = Value(-1.0)
w2 = Value(0.5)
b = Value(0.25)

act = w1 * x1 + w2 * x2 + b
n = act.relu()

print('act.data =', act.data)
print('n.data =', n.data)

n.backward()
print('w1.grad =', w1.grad)
print('w2.grad =', w2.grad)
print('b.grad  =', b.grad)

act.data = -3.25
n.data = 0
w1.grad = 0.0
w2.grad = 0.0
b.grad  = 0


观察：如果 `act.data < 0`，ReLU 输出 0，这个神经元这次不会把梯度传回去。现在把 `b` 改成 `4.0` 再运行上一格，看看梯度是否出现。

## 2. 看 `Neuron.__call__`

`Neuron` 的核心就在 `__call__`：对象像函数一样被调用。

In [4]:
print(inspect.getsource(Neuron.__call__))

    def __call__(self, x):
        act = sum((wi*xi for wi,xi in zip(self.w, x)), self.b)
        return act.relu() if self.nonlin else act



把源码翻译成人话：

```text
for 每个输入 xi 和对应权重 wi：
    计算 wi * xi
把这些乘积加起来
再加 b
如果 nonlin=True，就过 ReLU
```

也就是：

$$
out = ReLU(w_1x_1 + w_2x_2 + ... + b)
$$

## 3. 用真正的 Neuron 跑一次

In [5]:
random.seed(42)
n = Neuron(2)
print(n)
print('parameters:', n.parameters())

x = [Value(2.0), Value(-3.0)]
out = n(x)
print('out =', out)

out.backward()
for i, p in enumerate(n.parameters()):
    print(f'param {i}: data={p.data:.4f}, grad={p.grad:.4f}')

ReLUNeuron(2)
parameters: [Value(data=0.2788535969157675, grad=0), Value(data=-0.9499784895546661, grad=0), Value(data=0, grad=0)]
out = Value(data=3.4076426624955336, grad=0)
param 0: data=0.2789, grad=2.0000
param 1: data=-0.9500, grad=-3.0000
param 2: data=0.0000, grad=1.0000


## 4. Layer 和 MLP 只是继续套娃

```text
Layer = 多个 Neuron
MLP   = 多个 Layer
```

先看参数数量。

In [6]:
random.seed(42)
model = MLP(2, [3, 1])
print(model)
print('number of parameters:', len(model.parameters()))

x = [Value(1.0), Value(2.0)]
y = model(x)
print('prediction:', y)

y.backward()
print('first 5 parameter grads:')
for p in model.parameters()[:5]:
    print(p)

MLP of [Layer of [ReLUNeuron(2), ReLUNeuron(2), ReLUNeuron(2)], Layer of [LinearNeuron(3)]]
number of parameters: 13
prediction: Value(data=-0.18422396391917406, grad=0)
first 5 parameter grads:
Value(data=0.2788535969157675, grad=0.0)
Value(data=-0.9499784895546661, grad=0.0)
Value(data=0, grad=0.0)
Value(data=-0.4499413632617615, grad=0.0)
Value(data=-0.5535785237023545, grad=0.0)


## 5. 过关小测

对于：

```python
MLP(2, [3, 1])
```

参数数量怎么算？

```text
第一层：3 个神经元，每个神经元有 2 个 w + 1 个 b，共 3*3=9
第二层：1 个神经元，输入来自上一层 3 个输出，所以 3 个 w + 1 个 b，共 4
总共：13
```

In [7]:
model = MLP(2, [3, 1])
assert len(model.parameters()) == 13
print('OK: parameter count =', len(model.parameters()))

OK: parameter count = 13


## 6. TODO Lab - 自己算参数数量

TODO：补全一个参数数量计算函数。

规则：如果上一层有 `nin` 个输入，这一层有 `nout` 个神经元：

```text
这一层参数数量 = nout * (nin + 1)
```

`+1` 是 bias。

In [8]:
def count_mlp_parameters(nin, nouts):
    # TODO: 返回 MLP(nin, nouts) 的参数数量
    return None


qa_check_count_mlp_parameters(count_mlp_parameters)

TODO: 请先实现 count_mlp_parameters，再运行本 cell。


<details>
<summary><strong>Show / Hide 提示</strong></summary>

每一层参数数量是 `nout * (current_nin + 1)`，其中 `+1` 是 bias。算完一层后，下一层的输入维度变成这一层的 `nout`。

</details>

<details>
<summary><strong>Show / Hide 答案</strong></summary>

```python
def count_mlp_parameters(nin, nouts):
    total = 0
    current_nin = nin
    for nout in nouts:
        total += nout * (current_nin + 1)
        current_nin = nout
    return total
```

例如 `MLP(2, [3, 1])`：第一层 `3*(2+1)=9`，第二层 `1*(3+1)=4`，总共 `13`。

</details>

## What To Remember

这一节记住四句话：

```text
1. 神经元不是新魔法，只是很多个 Value 运算拼出来。
2. 参数就是 w 和 b，它们也是 Value。
3. forward 后得到预测值，backward 后每个参数都有 grad。
4. MLP = Layer 的列表，Layer = Neuron 的列表。
```

下一节才开始训练，也就是用 `p.data += -learning_rate * p.grad` 更新参数。

<!-- xiao-post:start -->
## 课后作业提交清单

这一节学完后，用下面 4 条自检：

```text
1. 我能写出一个神经元的公式：sum(w*x)+b，再接 ReLU。
2. 我能说清楚 Neuron、Layer、MLP 三层关系。
3. 我能手算 MLP(2, [3, 1]) 的参数个数。
4. 我能指出哪些 Value 是参数，哪些 Value 是输入。
```


## AI 复盘检查 Prompt

```text
你是我的 micrograd 神经网络结构检查官。
我刚学完 Neuron、Layer、MLP 的关系。
请你一次只问一个问题，检查我是否理解：
1. 神经元公式 w*x+b 是什么
2. 为什么 w 和 b 是参数
3. backward 后参数 grad 表示什么
4. MLP(2, [3, 1]) 的参数数量怎么算
如果我答错，请用一个 2 输入、1 神经元的小例子提示我。
```